In [1]:
# ---------------------------------------------------------
# Completeness evaluation setup
# ---------------------------------------------------------
# Completeness measures:
#
# "How much of the clinically important information from
# the original patient record is captured by the generated
# summary?"
#
# Evaluation direction:
#
# Original record -> reference checklist -> generated summary
#
# The reference checklist is created ONCE per patient and
# reused across all four summarization workflows.

import pandas as pd
import json
from pathlib import Path

In [2]:
# ---------------------------------------------------------
# Project paths
# ---------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent.parent

RESULTS_DIR = PROJECT_ROOT / "data" / "results"

EVALUATION_DIR = RESULTS_DIR / "evaluation"

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
# Files that will eventually store completeness outputs.
CHECKLIST_PATH = (
    EVALUATION_DIR / "completeness_reference_checklists.json"
)

COMPLETENESS_RESULTS_PATH = (
    EVALUATION_DIR / "completeness_results.json"
)


print("Project root:", PROJECT_ROOT)
print("Evaluation directory:", EVALUATION_DIR)
print("Checklist output:", CHECKLIST_PATH)

Project root: /Users/pallavi_chandanshive/projects/clinical-summarization-eval
Evaluation directory: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/results/evaluation
Checklist output: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/results/evaluation/completeness_reference_checklists.json


In [4]:
# ---------------------------------------------------------
# Load and reconstruct the frozen source clinical records
# ---------------------------------------------------------
# We use the exact preprocessing decisions from the four
# summarization workflows:
#
# 1. Remove invalid "#NAME?" notes.
# 2. Sort notes chronologically.
# 3. Remove duplicate note text within each patient.
#
# These deduplicated notes form the source of truth from
# which the completeness reference checklist will be built.

CLINICAL_NOTES_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "clinical_notes.csv"
)

notes = pd.read_csv(CLINICAL_NOTES_PATH)


# Remove invalid notes.
notes_clean = notes[
    notes["clean_note_text"]
    .astype(str)
    .str.strip() != "#NAME?"
].copy()


In [6]:
# Apply the same frozen deduplication used in the study.
notes_dedup = (
    notes_clean
    .sort_values([
        "person_id",
        "creation_timestamp"
    ])
    .drop_duplicates(
        subset=[
            "person_id",
            "clean_note_text"
        ],
        keep="first"
    )
    .reset_index(drop=True)
)

In [7]:
# ---------------------------------------------------------
# Reconstruct complete longitudinal records
# ---------------------------------------------------------

patient_source_documents = (
    notes_dedup
    .groupby("person_id", sort=False)["clean_note_text"]
    .apply(
        lambda texts: "\n\n".join(
            texts.astype(str)
        )
    )
    .to_dict()
)

In [8]:
# ---------------------------------------------------------
# Validate frozen source corpus
# ---------------------------------------------------------

print("Original notes:", len(notes))
print("After #NAME? removal:", len(notes_clean))
print("After deduplication:", len(notes_dedup))
print("Patients:", len(patient_source_documents))

Original notes: 1602
After #NAME? removal: 1595
After deduplication: 1103
Patients: 50


In [10]:
# ---------------------------------------------------------
# Load and validate the frozen study cohort
# ---------------------------------------------------------
# Completeness must evaluate the exact same 50 patients
# used by all four summarization workflows.

STUDY_PATIENT_IDS_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "patients"
    / "study_patient_ids.json"
)


with open(STUDY_PATIENT_IDS_PATH, "r") as f:
    study_patient_ids = json.load(f)


# Convert to sets only for validation.
study_patient_id_set = set(study_patient_ids)
source_patient_id_set = set(patient_source_documents.keys())


print("Study patients:", len(study_patient_ids))
print("Source patients:", len(patient_source_documents))

print(
    "Patient IDs match:",
    study_patient_id_set == source_patient_id_set
)

print(
    "Missing source records:",
    study_patient_id_set - source_patient_id_set
)

print(
    "Extra source records:",
    source_patient_id_set - study_patient_id_set
)

Study patients: 50
Source patients: 50
Patient IDs match: True
Missing source records: set()
Extra source records: set()


In [21]:
# ---------------------------------------------------------
# Reference checklist extraction prompt
# ---------------------------------------------------------
# The checklist represents clinically important information
# from the complete longitudinal source record.
#
# It is created from SOURCE NOTES ONLY. Generated summaries
# are never shown during checklist creation, preventing the
# four workflows from influencing the reference standard.

CHECKLIST_SYSTEM_PROMPT = """
You are evaluating longitudinal clinical summaries.

Your task is to identify the clinically important information from a
patient's longitudinal clinical record that a high-quality clinical
summary should reasonably preserve.

Use only information explicitly supported by the provided clinical record.

Do not attempt to reproduce every detail in the notes. 

Include a checklist item only if omitting that information would
meaningfully reduce the clinical completeness of a longitudinal summary.
Do not include routine physical examination findings, normal findings,
minor symptoms, or low-value details unless they materially affect the
diagnosis, treatment, clinical course, or outcome.

Exclude:
- administrative or clerical information
- repeated information that adds no new clinical meaning
- trivial details that would not reasonably be expected in a longitudinal summary
- unsupported assumptions or inferred diagnoses

Include clinically important information such as:
- major diagnoses and clinically significant conditions
- important procedures or interventions
- important treatments and medications when clinically relevant
- significant investigations and clinically meaningful results
- major changes in clinical status or disease progression
- complications or important adverse events
- important outcomes, recovery, follow-up, or disposition

Each checklist item must:
- represent one clinically meaningful fact or event
- be understandable independently
- contain enough context to evaluate whether a generated summary captured it
- avoid combining unrelated clinical facts into one item

Return ONLY valid JSON using this structure:

{
  "facts": [
    {
      "fact_id": 1,
      "fact": "Clinically important fact stated clearly and concisely.",
      "category": "diagnosis"
    }
  ]
}

Allowed categories:
diagnosis
procedure
treatment
investigation
clinical_course
complication
outcome
""".strip()


CHECKLIST_USER_PROMPT = """
Extract the clinically important reference facts from the following
longitudinal clinical record.

The resulting checklist will later be used to evaluate whether different
generated summaries captured the important information in this record.

LONGITUDINAL CLINICAL RECORD:

{patient_record}
""".strip()

In [22]:
from openai import OpenAI

# ---------------------------------------------------------
# OpenAI API client
# ---------------------------------------------------------
# The client is used to send requests to the evaluator model.
# It reads the OPENAI_API_KEY from the environment.

client = OpenAI()

EVALUATOR_MODEL = "gpt-5.4-mini"

In [23]:
# ---------------------------------------------------------
# Select ONE patient for pilot testing
# ---------------------------------------------------------
# We inspect the resulting checklist before running the
# extraction across the full 50-patient cohort.

test_person_id = study_patient_ids[0]

test_patient_record = patient_source_documents[
    test_person_id
]

print("Test patient:", test_person_id)
print("Record characters:", len(test_patient_record))

Test patient: 028998ee-babc-4096-9b28-001bc2f9a84e
Record characters: 13094


In [24]:
# ---------------------------------------------------------
# Pilot: extract reference checklist for ONE patient
# ---------------------------------------------------------
# Only one API call is made here.
#
# We will inspect the resulting checklist manually before
# running extraction across all 50 patients.


response = client.responses.create(
    model=EVALUATOR_MODEL,
    input=[
        {
            "role": "system",
            "content": CHECKLIST_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": CHECKLIST_USER_PROMPT.format(
                patient_record=test_patient_record
            )
        }
    ]
)

In [25]:
# Extract the model's JSON response.
raw_checklist = response.output_text

test_checklist = json.loads(raw_checklist)

In [28]:
# Assign deterministic fact IDs after extraction rather than
# relying on the LLM to generate sequential identifiers.

for fact_id, fact in enumerate(
    test_checklist["facts"],
    start=1
):
    fact["fact_id"] = fact_id


print("Patient:", test_person_id)
print("Number of reference facts:", len(test_checklist["facts"]))

print("\nReference checklist:\n")

for fact in test_checklist["facts"]:
    print(
        f'{fact["fact_id"]}. '
        f'[{fact["category"]}] '
        f'{fact["fact"]}'
    )

Patient: 028998ee-babc-4096-9b28-001bc2f9a84e
Number of reference facts: 12

Reference checklist:

1. [diagnosis] 15-year-old male presented to the ED with a 3-day history of severe lower abdominal pain and constipation.
2. [diagnosis] Initial ED impression was constipation, later associated with mild dehydration.
3. [investigation] Abdominal X-ray showed significant faecal loading with no evidence of obstruction or perforation.
4. [investigation] Blood tests including FBC, U&Es, CRP, and LFTs were normal.
5. [treatment] The patient was treated with IV 0.9% sodium chloride for mild dehydration.
6. [treatment] A single 10 mg oral dose of bisacodyl was administered in the ED to initiate bowel movement.
7. [treatment] Oral macrogol/Movicol was started for constipation during admission and later discontinued once symptoms resolved.
8. [procedure] The patient was referred and admitted to paediatrics for supportive management and observation.
9. [procedure] Physiotherapy was provided to enco

In [29]:
# ---------------------------------------------------------
# Inspect pilot checklist
# ---------------------------------------------------------

print("Patient:", test_person_id)
print("Number of reference facts:", len(test_checklist["facts"]))

print("\nReference checklist:\n")

for fact in test_checklist["facts"]:
    print(
        f'{fact["fact_id"]}. '
        f'[{fact["category"]}] '
        f'{fact["fact"]}'
    )

Patient: 028998ee-babc-4096-9b28-001bc2f9a84e
Number of reference facts: 12

Reference checklist:

1. [diagnosis] 15-year-old male presented to the ED with a 3-day history of severe lower abdominal pain and constipation.
2. [diagnosis] Initial ED impression was constipation, later associated with mild dehydration.
3. [investigation] Abdominal X-ray showed significant faecal loading with no evidence of obstruction or perforation.
4. [investigation] Blood tests including FBC, U&Es, CRP, and LFTs were normal.
5. [treatment] The patient was treated with IV 0.9% sodium chloride for mild dehydration.
6. [treatment] A single 10 mg oral dose of bisacodyl was administered in the ED to initiate bowel movement.
7. [treatment] Oral macrogol/Movicol was started for constipation during admission and later discontinued once symptoms resolved.
8. [procedure] The patient was referred and admitted to paediatrics for supportive management and observation.
9. [procedure] Physiotherapy was provided to enco

In [30]:
# ---------------------------------------------------------
# Extract reference completeness checklists for all patients
# ---------------------------------------------------------
# One LLM call is made per patient.
#
# Results are saved after every patient so that:
# - progress is not lost if execution stops
# - rerunning this cell skips patients already completed


# ---------------------------------------------------------
# Load existing progress if available
# ---------------------------------------------------------

if CHECKLIST_PATH.exists():
    with open(CHECKLIST_PATH, "r") as f:
        reference_checklists = json.load(f)

    print(
        "Loaded existing checklists:",
        len(reference_checklists)
    )
else:
    reference_checklists = {}


In [31]:
# ---------------------------------------------------------
# Keep our already-validated pilot result
# ---------------------------------------------------------
# This avoids making another API call for the pilot patient.

if test_person_id not in reference_checklists:
    reference_checklists[test_person_id] = {
        "person_id": test_person_id,
        "facts": test_checklist["facts"]
    }

    with open(CHECKLIST_PATH, "w") as f:
        json.dump(
            reference_checklists,
            f,
            indent=2
        )

In [32]:

# ---------------------------------------------------------
# Extract checklist for each remaining patient
# ---------------------------------------------------------

for i, person_id in enumerate(
    study_patient_ids,
    start=1
):

    # Skip patients already completed.
    if person_id in reference_checklists:
        print(
            f"[{i}/50] {person_id} - already completed"
        )
        continue

    patient_record = patient_source_documents[person_id]

    print(
        f"[{i}/50] Extracting checklist for {person_id}..."
    )

    response = client.responses.create(
        model=EVALUATOR_MODEL,
        input=[
            {
                "role": "system",
                "content": CHECKLIST_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": CHECKLIST_USER_PROMPT.format(
                    patient_record=patient_record
                )
            }
        ]
    )

    # Parse the model's JSON response.
    checklist = json.loads(response.output_text)

    # Assign deterministic fact IDs in Python.
    for fact_id, fact in enumerate(
        checklist["facts"],
        start=1
    ):
        fact["fact_id"] = fact_id

    # Store this patient's checklist.
    reference_checklists[person_id] = {
        "person_id": person_id,
        "facts": checklist["facts"]
    }

    # Save immediately after each patient.
    with open(CHECKLIST_PATH, "w") as f:
        json.dump(
            reference_checklists,
            f,
            indent=2
        )

    print(
        f"       Saved {len(checklist['facts'])} facts"
    )

print("\nChecklist extraction complete.")
print(
    "Patients completed:",
    len(reference_checklists)
)
print(
    "Saved to:",
    CHECKLIST_PATH
)

[1/50] 028998ee-babc-4096-9b28-001bc2f9a84e - already completed
[2/50] Extracting checklist for 04df53ea-55c1-48d9-84a1-1f15c133b29b...
       Saved 14 facts
[3/50] Extracting checklist for 05192757-942f-460d-b4ff-004ec39cc5ee...
       Saved 20 facts
[4/50] Extracting checklist for 0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf...
       Saved 14 facts
[5/50] Extracting checklist for 0f438665-d430-4adb-8acc-c3beed9e4942...
       Saved 13 facts
[6/50] Extracting checklist for 136c7916-4f9b-4e5c-bf01-77e9d2c681a2...
       Saved 11 facts
[7/50] Extracting checklist for 137b8481-4f1d-4b7f-babd-20f7117023ad...
       Saved 14 facts
[8/50] Extracting checklist for 1705dd0f-011a-492c-b006-b27e03f2f4ed...
       Saved 16 facts
[9/50] Extracting checklist for 1dbe23dc-0d1e-431b-81eb-497282b46a14...
       Saved 17 facts
[10/50] Extracting checklist for 28570119-9cdc-4120-98c0-4edb76cf36a3...
       Saved 12 facts
[11/50] Extracting checklist for 29ea304f-821d-474e-81a1-394ca3945e02...
       Saved 9 f

In [33]:
# ---------------------------------------------------------
# Validate completeness reference checklists
# ---------------------------------------------------------

# Number of patients
print("Patients with checklists:", len(reference_checklists))


# Number of facts per patient
fact_counts = {
    person_id: len(data["facts"])
    for person_id, data in reference_checklists.items()
}

print("Total reference facts:", sum(fact_counts.values()))
print("Minimum facts per patient:", min(fact_counts.values()))
print("Maximum facts per patient:", max(fact_counts.values()))
print(
    "Average facts per patient:",
    round(sum(fact_counts.values()) / len(fact_counts), 2)
)

Patients with checklists: 50
Total reference facts: 703
Minimum facts per patient: 9
Maximum facts per patient: 21
Average facts per patient: 14.06


In [34]:
# ---------------------------------------------------------
# Check that all 50 study patients are present
# ---------------------------------------------------------

checklist_patient_ids = set(reference_checklists.keys())

print(
    "Patient IDs match:",
    checklist_patient_ids == study_patient_id_set
)

print(
    "Missing patients:",
    study_patient_id_set - checklist_patient_ids
)

print(
    "Extra patients:",
    checklist_patient_ids - study_patient_id_set
)

Patient IDs match: True
Missing patients: set()
Extra patients: set()


In [35]:
# ---------------------------------------------------------
# Validate fact structure and categories
# ---------------------------------------------------------

allowed_categories = {
    "diagnosis",
    "procedure",
    "treatment",
    "investigation",
    "clinical_course",
    "complication",
    "outcome"
}

invalid_facts = []

for person_id, data in reference_checklists.items():

    for fact in data["facts"]:

        if (
            "fact_id" not in fact
            or "fact" not in fact
            or "category" not in fact
            or fact["category"] not in allowed_categories
        ):
            invalid_facts.append({
                "person_id": person_id,
                "fact": fact
            })


print("Invalid facts:", len(invalid_facts))

Invalid facts: 0
